# 🦙 Llama-3.1-8B-Stheno-v3.4 on Free Google Colab + Ngrok Tunnel for ZetaForge

This notebook configures a free Google Colab runtime to:
1. Verify and utilize the free **NVIDIA T4 GPU** (16 GB VRAM).
2. Fast-download the **Llama-3.1-8B-Stheno-v3.4-Q4_K_M GGUF** (~4.92 GB) using multi-threaded `aria2c`.
3. Run **KoboldCpp** (CUDA accelerated) with 100% GPU layer offloading and Flash Attention.
4. Bind your reserved static Ngrok domain (`https://paralegal-pampers-chevron.ngrok-free.dev/`).
5. Serve both an **OpenAI-compatible API (`/v1`)** and **Kobold API (`/api/v1`)** for ZetaForge and other frontends.

---
### ⚠️ Before Running:
- Ensure your runtime is set to GPU: Navigate to **Runtime ➔ Change runtime type ➔ T4 GPU ➔ Save**.
- Have your **Ngrok Authtoken** ready from [ngrok.com/dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).

In [ ]:
#@title 1. Check GPU Environment
#@markdown Run this cell to make sure a CUDA GPU is attached to your free Colab session.

import torch
import sys

if not torch.cuda.is_available():
    raise SystemError("❌ No GPU detected! Please go to Runtime -> Change runtime type -> Select 'T4 GPU' and re-run.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"✅ Hardware Accelerated: {gpu_name} ({vram_gb:.2f} GB VRAM detected)")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
#@title 2. Install Tools & Download KoboldCpp Engine
#@markdown Installs multi-threaded downloaders, pyngrok, and the latest pre-compiled KoboldCpp Linux CUDA binary.

print("📦 Installing aria2c and Python utilities...")
!apt-get update -y -qq && apt-get install -y -qq aria2
!pip install -q pyngrok requests huggingface_hub

print("⚙️ Downloading KoboldCpp Linux x64 CUDA binary...")
!curl -fLo koboldcpp https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64 && chmod +x koboldcpp

print("✅ KoboldCpp engine ready!")

In [ ]:
#@title 3. Download Llama-3.1-8B-Stheno-v3.4-Q4_K_M
#@markdown Pulls the Q4_K_M GGUF directly from the official Bartowski repository.

import os

MODEL_REPO_URL = "https://huggingface.co/bartowski/Llama-3.1-8B-Stheno-v3.4-GGUF/resolve/main/Llama-3.1-8B-Stheno-v3.4-Q4_K_M.gguf"
MODEL_FILENAME = "Llama-3.1-8B-Stheno-v3.4-Q4_K_M.gguf"
MODEL_PATH = f"/content/{MODEL_FILENAME}"

if not os.path.exists(MODEL_PATH):
    print(f"📥 Downloading {MODEL_FILENAME} (~4.92 GB)... This takes ~45-90 seconds on Colab.")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{MODEL_REPO_URL}" -d /content -o "{MODEL_FILENAME}"
    print("\n✅ Model downloaded successfully!")
else:
    print(f"✅ Model file already present at {MODEL_PATH}")

In [ ]:
#@title 4. Configure Ngrok Tunnel & Domain (Auto-Clears Stale Tunnels)
#@markdown Enter your Ngrok Authtoken below. The cell will disconnect any orphan endpoints first.

import getpass
import time
import subprocess
import requests
from pyngrok import conf, ngrok

NGROK_AUTHTOKEN = "" #@param {type:"string"}
RESERVED_DOMAIN = "paralegal-pampers-chevron.ngrok-free.dev" #@param {type:"string"}
LOCAL_PORT = 5001

# Retrieve token if stored in Colab Secrets
if not NGROK_AUTHTOKEN:
    try:
        from google.colab import userdata
        NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
    except Exception:
        pass

if not NGROK_AUTHTOKEN:
    NGROK_AUTHTOKEN = getpass.getpass("🔑 Paste your Ngrok Authtoken: ")

token_clean = NGROK_AUTHTOKEN.strip()
domain_clean = RESERVED_DOMAIN.strip().replace("https://", "").replace("/", "")

# 1. Kill any existing local processes on the VM
print("🧹 Terminating any stale local ngrok processes...")
try:
    ngrok.kill()
except Exception:
    pass
subprocess.run(["pkill", "-9", "ngrok"], stderr=subprocess.DEVNULL)
time.sleep(2)

# 2. Query Ngrok Cloud API to stop any ghost tunnel holding this domain
print("🔍 Checking for orphaned remote endpoints via Ngrok Cloud API...")
api_headers = {
    "Authorization": f"Bearer {token_clean}",
    "Ngrok-Version": "2"
}

try:
    endpoints_res = requests.get("https://api.ngrok.com/endpoints", headers=api_headers, timeout=10)
    if endpoints_res.status_code == 200:
        endpoints_data = endpoints_res.json().get("endpoints", [])
        for ep in endpoints_data:
            if domain_clean in ep.get("public_url", ""):
                ep_id = ep.get("id")
                print(f"⚠️ Found active remote session {ep_id} bound to {domain_clean}. Stopping it...")
                requests.delete(f"https://api.ngrok.com/endpoints/{ep_id}", headers=api_headers, timeout=10)
                time.sleep(3)
except Exception as e:
    print(f"(Notice: Remote endpoint inspection skipped: {e})")

# 3. Set auth token and connect tunnel
conf.get_default().auth_token = token_clean

tunnel = None
try:
    tunnel = ngrok.connect(LOCAL_PORT, domain=domain_clean)
except Exception as err:
    print(f"Standard connection notice: {err}")
    print("Retrying with connection pooling enabled (--pooling-enabled)...")
    time.sleep(2)
    # Fallback: pooling bypasses ERR_NGROK_334 if edge release is delayed
    tunnel = ngrok.connect(LOCAL_PORT, domain=domain_clean, pooling_enabled=True)

print("=" * 75)
print("🚀 NGROK TUNNEL IS ACTIVE!")
print(f"🌐 Public Hostname    : {tunnel.public_url}")
print(f"🔗 OpenAI API Base URL: {tunnel.public_url}/v1")
print(f"🔗 Kobold API Base URL: {tunnel.public_url}/api/v1")
print("=" * 75)
print("👉 In ZetaForge, point your OpenAI-compatible API base URL to:")
print(f"   {tunnel.public_url}/v1")
print("=" * 75)

In [ ]:
#@title 5. Launch KoboldCpp Server (Runs in Foreground)
#@markdown Keep this cell running while you chat/generate in ZetaForge. Logs will stream here.

CONTEXT_SIZE = 8192 #@param [4096, 8192, 16384] {type:"raw"}
GPU_LAYERS = 33 # 33 offloads all layers of Llama-3.1-8B into T4's 16GB VRAM

print(f"🔥 Starting server with context={CONTEXT_SIZE} on port 5001...")
print("Press the Stop button on this cell whenever you want to shut down the server.\n")

!./koboldcpp --model /content/Llama-3.1-8B-Stheno-v3.4-Q4_K_M.gguf \
    --port 5001 \
    --gpulayers {GPU_LAYERS} \
    --contextsize {CONTEXT_SIZE} \
    --flashattention \
    --smartcontext

In [ ]:
#@title 6. (Optional) Quick API Verification Test
#@markdown You can run this in a separate session or cell to test completion responses from your tunnel.

import requests
import json

API_ENDPOINT = "https://paralegal-pampers-chevron.ngrok-free.dev/v1/chat/completions"

headers = {
    "Content-Type": "application/json",
    "ngrok-skip-browser-warning": "true" # Prevents ngrok free tier HTML interstitial
}

payload = {
    "model": "Llama-3.1-8B-Stheno-v3.4",
    "messages": [
        {"role": "system", "content": "You are a creative and expressive assistant."},
        {"role": "user", "content": "Write a single sentence greeting introducing yourself."}
    ],
    "temperature": 0.7,
    "max_tokens": 80
}

try:
    response = requests.post(API_ENDPOINT, headers=headers, json=payload, timeout=60)
    if response.status_code == 200:
        reply = response.json()["choices"][0]["message"]["content"]
        print("\n🤖 Model Output:\n", reply)
    else:
        print(f"❌ Server returned status {response.status_code}:", response.text)
except Exception as err:
    print("❌ Connection error (ensure Cell 5 is running):", err)